# TinyECG: arrhythmia classification with a ~1.5k-parameter 1D CNN

Classifies single ECG heartbeats (187 samples each) into five classes: Fusion (F), Normal (N), Unknown (Q), Supraventricular ectopic (S) and Ventricular ectopic (V).

The goal is a model that is accurate **and** tiny. Models are compared with a composite objective:

`score = 0.7 * accuracy + 0.3 * F2(macro) - 0.02 * ln(P)`, where P is the number of trainable parameters.

Pipeline:
1. **Sweep** several model widths on a stratified validation split (with per-class bias tuning).
2. **Pick** the width with the best composite score.
3. **Export** the model weights and, optionally, predictions for an unlabeled test file.

Only the provided training data is used; no external data or pretrained weights.

In [1]:
import os, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, fbeta_score, classification_report
from sklearn.model_selection import train_test_split

DATA_DIR = "data"            # folder with train.csv (and optionally test.csv)
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")   # columns: ecg_0 ... ecg_186, type (0-4)
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")    # optional: unlabeled beats (ecg_* columns, optional Id)
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
BATCH = 256
LR = 3e-3
CLASS_POWER = 0.1
WIDTHS = [4, 6, 8]     # widths to compare in the sweep
SWEEP_EPOCHS = 80           # epochs per run (final retrain uses the same number)
USE_FULL_DATA = False     # retrain best width on 100% of train.csv for the final model
USE_BIAS = True
USE_EMA = True              # exponential moving average of weights (smooths epoch-to-epoch noise, 0 extra params)
EMA_DECAY = 0.998
USE_DIFF = True             # add first-difference channel (slope of the beat) as a 2nd input channel (+7*width params)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

device: cpu


## 1. Data

In [2]:
def prep(x):
    """Per-beat z-score; optionally stack a z-scored first-difference channel -> (N, C, 187)."""
    x = x.astype(np.float32)
    x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6)
    if USE_DIFF:
        d = np.diff(x, axis=1, prepend=x[:, :1])
        d = d / (d.std(1, keepdims=True) + 1e-6)
        return np.stack([x, d], axis=1).astype(np.float32)
    return x[:, None, :]
IN_CH = 2 if USE_DIFF else 1

tr = pd.read_csv(TRAIN_CSV)
feat = [c for c in tr.columns if c.startswith("ecg_")]
X = prep(tr[feat].values)
y = tr["type"].values.astype(np.int64)

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
Xva_t = torch.tensor(Xva)
print("train:", Xtr.shape, "val:", Xva.shape, "| val class counts:", np.bincount(yva))

train: (70043, 2, 187) val: (17511, 2, 187) | val class counts: [  128 14494  1286   445  1158]


## 2. Model and helpers

In [3]:
class DWBlock(nn.Module):
    def __init__(self, cin, cout, k=5, pool=2):
        super().__init__()
        self.dw = nn.Conv1d(cin, cin, k, padding=k // 2, groups=cin, bias=False)
        self.pw = nn.Conv1d(cin, cout, 1, bias=False)
        self.bn = nn.BatchNorm1d(cout)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool1d(pool) if pool > 1 else nn.Identity()
    def forward(self, x):
        return self.pool(self.act(self.bn(self.pw(self.dw(x)))))

class TinyECG(nn.Module):
    def __init__(self, w=8, n_classes=5, in_ch=None):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch or IN_CH, w, 7, padding=3, bias=False), nn.BatchNorm1d(w), nn.ReLU(), nn.MaxPool1d(2))
        self.b1 = DWBlock(w, 2 * w)
        self.b2 = DWBlock(2 * w, 4 * w)
        self.b3 = DWBlock(4 * w, 4 * w, pool=1)
        self.fc = nn.Linear(4 * w, n_classes)
    def forward(self, x):
        x = self.b3(self.b2(self.b1(self.stem(x))))
        return self.fc(x.mean(-1))

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def comp_score(y_true, y_pred, P, average="macro"):
    acc = accuracy_score(y_true, y_pred)
    f2 = fbeta_score(y_true, y_pred, beta=2, average=average, zero_division=0)
    return 0.7 * acc + 0.3 * f2 - 0.02 * math.log(P), acc, f2

@torch.no_grad()
def get_logits(model, Xt, bs=2048):
    model.eval()
    return torch.cat([model(Xt[i:i + bs].to(dev)).cpu() for i in range(0, len(Xt), bs)]).numpy()

def augment(xb):
    xb = xb * (1 + 0.1 * torch.randn(xb.size(0), 1, 1, device=xb.device))
    xb = xb + 0.03 * torch.randn_like(xb)
    return torch.roll(xb, random.randint(-4, 4), dims=-1)

def train_model(width, epochs, X_tr, y_tr, X_va=None, y_va=None, class_power=CLASS_POWER, seed=SEED):
    """Train one model. With validation data: keep the best-epoch weights. Without: keep the last epoch.
    If USE_EMA, the kept/evaluated weights are the exponential moving average of the training weights."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = TinyECG(width).to(dev)
    eval_model = TinyECG(width).to(dev)
    P = n_params(model)
    counts = np.bincount(y_tr, minlength=5)
    cw = (len(y_tr) / (5 * counts)) ** class_power
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=dev))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * math.ceil(len(Xt) / BATCH))
    ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
    step, best, best_state = 0, -1e9, None
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            xb, yb = Xt[idx].to(dev), yt[idx].to(dev)
            loss = loss_fn(model(augment(xb)), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            step += 1
            d = min(EMA_DECAY, (1 + step) / (10 + step))
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    ema[k].mul_(d).add_(v.detach(), alpha=1 - d)
                else:
                    ema[k].copy_(v)
        cur_state = ema if USE_EMA else model.state_dict()
        if X_va is not None:
            eval_model.load_state_dict(cur_state)
            s_, acc, f2 = comp_score(y_va, get_logits(eval_model, X_va).argmax(1), P)
            if s_ > best:
                best = s_
                best_state = {k: v.detach().clone() for k, v in cur_state.items()}
            if ep % 10 == 0 or ep == epochs:
                print(f"   w={width} ep{ep:02d} val_acc={acc:.4f} F2={f2:.4f} score={s_:.4f}")
    final_state = best_state if best_state is not None else {k: v.detach().clone() for k, v in (ema if USE_EMA else model.state_dict()).items()}
    model.load_state_dict(final_state)
    return model, P

def tune_bias(logits, y_true, P, grid=np.linspace(-2, 2, 41), rounds=3):
    """Per-class logit offsets maximizing the competition score (coordinate ascent)."""
    bias = np.zeros(5)
    f = lambda b: comp_score(y_true, (logits + b).argmax(1), P)[0]
    before = cur = f(bias)
    for _ in range(rounds):
        for k in range(5):
            for d in grid:
                bt = bias.copy(); bt[k] = d
                sc = f(bt)
                if sc > cur + 1e-5:
                    cur, bias = sc, bt
    return bias, before, cur

## 3. Width sweep (validation only)

In [4]:
results, models = [], {}
for w in WIDTHS:
    t0 = time.time()
    m, P = train_model(w, SWEEP_EPOCHS, Xtr, ytr, Xva_t, yva)
    lv = get_logits(m, Xva_t)
    if USE_BIAS:
        bias, before, after = tune_bias(lv, yva, P)
    else:
        bias = np.zeros(5); before = after = comp_score(yva, lv.argmax(1), P)[0]
    _, acc, f2 = comp_score(yva, (lv + bias).argmax(1), P)
    results.append(dict(width=w, params=P, val_acc=acc, val_f2=f2, score_before_bias=before, score_after_bias=after,
                        minutes=(time.time() - t0) / 60))
    models[w] = (m, bias)
    print(f"width {w}: params={P} score {before:.4f} -> {after:.4f}")

res = pd.DataFrame(results).round(4)
print(res.to_string(index=False))
best_w = int(res.loc[res["score_after_bias"].idxmax(), "width"])
print("\nBest width by validation score:", best_w)

   w=4 ep10 val_acc=0.9282 F2=0.5147 score=0.6709
   w=4 ep20 val_acc=0.9563 F2=0.7284 score=0.7546
   w=4 ep30 val_acc=0.9615 F2=0.7900 score=0.7767
   w=4 ep40 val_acc=0.9661 F2=0.8157 score=0.7877
   w=4 ep50 val_acc=0.9676 F2=0.8277 score=0.7923
   w=4 ep60 val_acc=0.9688 F2=0.8376 score=0.7961
   w=4 ep70 val_acc=0.9687 F2=0.8403 score=0.7969
   w=4 ep80 val_acc=0.9692 F2=0.8395 score=0.7969
width 4: params=785 score 0.7979 -> 0.7995
   w=6 ep10 val_acc=0.9402 F2=0.5414 score=0.6745
   w=6 ep20 val_acc=0.9664 F2=0.8112 score=0.7738
   w=6 ep30 val_acc=0.9725 F2=0.8483 score=0.7891
   w=6 ep40 val_acc=0.9752 F2=0.8573 score=0.7937
   w=6 ep50 val_acc=0.9772 F2=0.8641 score=0.7972
   w=6 ep60 val_acc=0.9774 F2=0.8676 score=0.7984
   w=6 ep70 val_acc=0.9781 F2=0.8693 score=0.7993
   w=6 ep80 val_acc=0.9778 F2=0.8702 score=0.7994
width 6: params=1487 score 0.7995 -> 0.8048
   w=8 ep10 val_acc=0.9483 F2=0.5808 score=0.6824
   w=8 ep20 val_acc=0.9697 F2=0.8401 score=0.7752
   w=8 ep30 v

## 4. Validation report for the best width

In [10]:
best_model, best_bias = models[best_w]
P = n_params(best_model)
pv = (get_logits(best_model, Xva_t) + best_bias).argmax(1)
print("bias:", np.round(best_bias, 2))
print(classification_report(yva, pv, target_names=["F(0)", "N(1)", "Q(2)", "SVEB(3)", "VEB(4)"], digits=4))

bias: [ 0.4 -0.4 -0.6  0.5 -0.5]
              precision    recall  f1-score   support

        F(0)     0.8080    0.7891    0.7984       128
        N(1)     0.9853    0.9912    0.9882     14494
        Q(2)     0.9836    0.9806    0.9821      1286
     SVEB(3)     0.8295    0.7213    0.7716       445
      VEB(4)     0.9472    0.9301    0.9386      1158

    accuracy                         0.9780     17511
   macro avg     0.9107    0.8824    0.8958     17511
weighted avg     0.9774    0.9780    0.9776     17511



## 5. Final model and export

If `USE_FULL_DATA = True`, the best width is retrained on **all** of train.csv (same epochs, last-epoch weights) and the sweep's bias offsets are applied. In my runs, keeping the best validation checkpoint (`USE_FULL_DATA = False`) did better than retraining on all the data, so it is off by default.

If a `test.csv` is present, predictions for it are also written.

In [ ]:
if USE_FULL_DATA:
    final_model, P = train_model(best_w, SWEEP_EPOCHS, X, y)
else:
    final_model, P = best_model, n_params(best_model)
final_bias = best_bias if USE_BIAS else np.zeros(5)
print(f"final model: width={best_w}, trainable params={P}, full_data={USE_FULL_DATA}")

RUN_TAG = f"w{best_w}_e{SWEEP_EPOCHS}{'_ema' if USE_EMA else ''}{'_diff' if USE_DIFF else ''}_{'full' if USE_FULL_DATA else 'split'}"
ckpt = os.path.join(OUT_DIR, f"model_{RUN_TAG}.pt")
torch.save(final_model.state_dict(), ckpt)
np.save(os.path.join(OUT_DIR, f"bias_{RUN_TAG}.npy"), final_bias)

# Optional: classify an unlabeled test file if one is available
if os.path.exists(TEST_CSV):
    te = pd.read_csv(TEST_CSV)
    pt = (get_logits(final_model, torch.tensor(prep(te[feat].values))) + final_bias).argmax(1)
    out = pd.DataFrame({"type": pt})
    if "Id" in te.columns:
        out.insert(0, "Id", te["Id"])
    pred_path = os.path.join(OUT_DIR, f"predictions_{RUN_TAG}.csv")
    out.to_csv(pred_path, index=False)
    print("wrote", pred_path, "| predicted class counts:", np.bincount(pt, minlength=5).tolist())
else:
    print("no test file found; skipping predictions")

## 6. Saved artifacts

In [ ]:
print("Saved to", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", f)
print("trainable parameters:", n_params(final_model))